# 18 — PC1 deep dive + final test-set validation (5,000-user cohort)

Continuation of notebook 17. Two things left unresolved there:

- **PC1** (68% of `H1`'s variance) was only checked evidence-by-evidence (each behavioral factor and each movie-side signal correlated separately against PC1). Never combined into one joint model, so we don't actually know how much of PC1 the *combination* of evidence explains.
- **Part 5d**: the four-way (or five-way, if PC1 yields a usable channel) test-set validation, comparing Real RBM alone / existing hand-picked personality channel / naive learned-bias baseline / the extremity channel trained in 17's Part 5c / (conditionally) a PC1-based channel.

Notebook 17 is left as-is for review. This notebook is self-contained (reloads raw data, recomputes `H1`/PCA — both fast and deterministic — rather than depending on 17's in-memory state).

## Part 0 — Setup

Reload what's needed: `W1`/`bh1` (frozen real part), `channelB1`/`mask`, cohort/vocab, `personality`/`user_means`, Part 3's `user_behavioral_factors_5k.csv` (already saved by notebook 17 — no recompute), and Part 5c's trained extremity channel (`rbmB2ext_weights_5k.npy`).

In [1]:
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import sparse, stats

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

data_dir = root / "data"
proc = root / "data" / "processed"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))


paths = {
    "W1": proc / "rbmB1_weights_5k.npy",
    "bh1": proc / "rbmB1_bias_hidden_5k.npy",
    "W2": proc / "rbmB2_weights_5k.npy",
    "bh2": proc / "rbmB2_bias_hidden_5k.npy",
    "W2ext": proc / "rbmB2ext_weights_5k.npy",
    "bh2ext": proc / "rbmB2ext_bias_hidden_5k.npy",
    "channelB1": proc / "channelB1_softmax.npy",
    "channelB2": proc / "channelB2_personality.npy",
    "mask": proc / "mask.npy",
    "cohort": proc / "cohort_user_ids.npy",
    "vocab": proc / "movie_vocab.npy",
    "means": proc / "user_means.npy",
    "personality": proc / "personality.npy",
    "mu_global": proc / "mu_global.npy",
    "test": proc / "test_labels.csv",
}
for name, p in paths.items():
    assert p.exists(), f"Missing {p} ({name})"

W1 = np.load(paths["W1"])
bh1 = np.load(paths["bh1"])
W2 = np.load(paths["W2"])
bh2 = np.load(paths["bh2"])
W2ext = np.load(paths["W2ext"])
bh2ext = np.load(paths["bh2ext"])
channelB1 = np.load(paths["channelB1"], mmap_mode="r")
channelB2 = np.load(paths["channelB2"], mmap_mode="r")
mask = np.load(paths["mask"], mmap_mode="r")
cohort_user_ids = np.load(paths["cohort"]).astype(int)
movie_vocab = np.load(paths["vocab"]).astype(int)
user_means = np.load(paths["means"]).astype(np.float64)
personality = np.load(paths["personality"]).astype(np.float64)
mu_global = float(np.load(paths["mu_global"]))
test_labels = pd.read_csv(paths["test"])

n_users, n_movies, k_b1 = channelB1.shape
n_hidden = bh1.shape[0]
assert k_b1 == K

user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}
vocab_set = set(int(m) for m in movie_vocab)

print(f"Project root: {root}")
print(f"n_users={n_users:,}  n_movies={n_movies:,}  n_hidden={n_hidden}")

# --- Load Part 3's user behavioral factors (already saved by notebook 17) ---
user_factors = pd.read_csv(out_dir / "user_behavioral_factors_5k.csv")
assert list(user_factors["userId"]) == list(cohort_user_ids), "row order must match cohort_user_ids"
FACTOR_NAMES = ["leniency", "extremity", "activity", "genre_entropy", "contrarian_bias", "popularity_bias"]
for f in FACTOR_NAMES:
    globals()[f] = user_factors[f].to_numpy()
print(f"Loaded user_behavioral_factors_5k.csv: {user_factors.shape}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
n_users=5,000  n_movies=13,129  n_hidden=128
Loaded user_behavioral_factors_5k.csv: (5000, 8)


## Part 1 — Recompute H1, PCA, and movie-side evidence

Fast + deterministic, so recomputing here (rather than reloading pickled intermediates from 17) keeps this notebook self-contained. Reuses the exact same logic as 17.

In [2]:
_t0 = time.time()

# H1 forward pass (frozen real part)
X1 = np.asarray(channelB1, dtype=np.float32).reshape(n_users, -1)
H1 = sigmoid(X1 @ W1 + bh1)

# Decode train-only ratings (needed for year/genome per-user profiles)
train_rows = []
step = 250
for i0 in range(0, n_users, step):
    i1 = min(i0 + step, n_users)
    m_block = np.asarray(mask[i0:i1])
    c_block = np.asarray(channelB1[i0:i1])
    for ii in range(i1 - i0):
        cols = np.where(m_block[ii] == 1)[0]
        if len(cols) == 0:
            continue
        ks = c_block[ii, cols, :].argmax(axis=1)
        ratings = RATING_LEVELS[ks]
        uid = int(cohort_user_ids[i0 + ii])
        mids = movie_vocab[cols]
        for mid, r in zip(mids, ratings):
            train_rows.append((uid, int(mid), float(r)))
train_df = pd.DataFrame(train_rows, columns=["userId", "movieId", "rating"])
assert len(train_df) == int(np.asarray(mask).sum())

# PCA on H1
from sklearn.decomposition import PCA
pca = PCA(n_components=5, random_state=0)
H1_centered = H1 - H1.mean(axis=0)
pcs = pca.fit_transform(H1_centered)
print(f"PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%  "
      f"PC3={pca.explained_variance_ratio_[2]*100:.1f}%")
pc1 = pcs[:, 0]

# --- Movie metadata (genres, year) ---
import re
movies_raw = pd.read_csv(data_dir / "movie.csv")
movies_raw = movies_raw[movies_raw["movieId"].isin(vocab_set)].copy()
YEAR_RE = re.compile(r"\((\d{4})\)\s*$")
movies_raw["year"] = movies_raw["title"].map(lambda t: int(m.group(1)) if (m := YEAR_RE.search(str(t))) else np.nan)
movie_meta = movies_raw.set_index("movieId")[["title", "genres", "year"]].to_dict("index")

year_by_movie = {int(m): movie_meta.get(int(m), {}).get("year", np.nan) for m in movie_vocab}
train_df["year"] = train_df["movieId"].map(year_by_movie)
user_avg_year = train_df.groupby("userId")["year"].mean().reindex(cohort_user_ids).to_numpy()

# --- Genome tags: per-user average tag-relevance profile (vectorized, same as nb17) ---
genome_tags_df = pd.read_csv(data_dir / "genome_tags.csv").sort_values("tagId")
genome_tag_ids = genome_tags_df["tagId"].to_numpy()
genome_tag_names = genome_tags_df["tag"].to_numpy()

genome_scores_df = pd.read_csv(data_dir / "genome_scores.csv")
genome_scores_df = genome_scores_df[genome_scores_df["movieId"].isin(vocab_set)]
genome_movie_ids = np.sort(genome_scores_df["movieId"].unique())
genome_pivot = genome_scores_df.pivot(index="movieId", columns="tagId", values="relevance").reindex(
    index=genome_movie_ids, columns=genome_tag_ids)
genome_matrix = genome_pivot.to_numpy(dtype=np.float64)
del genome_scores_df, genome_pivot

genome_row_by_movie = {int(mid): i for i, mid in enumerate(genome_movie_ids)}
train_genome = train_df[train_df["movieId"].isin(genome_row_by_movie)].copy()
train_genome["grow"] = train_genome["movieId"].map(genome_row_by_movie).astype(int)
train_genome["urow"] = train_genome["userId"].map(user_to_row).astype(int)
ones = np.ones(len(train_genome))
ind = sparse.csr_matrix((ones, (train_genome["urow"], train_genome["grow"])), shape=(n_users, len(genome_movie_ids)))
row_sums = np.asarray(ind.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
ind_norm = ind.multiply(1.0 / row_sums[:, None]).tocsr()
user_genome_profile = np.asarray(ind_norm @ genome_matrix)  # (n_users, n_tags)

print(f"Rebuilt H1, PCA, year, and genome-tag evidence in {time.time()-_t0:.1f}s")

PC1=68.0%  PC2=25.6%  PC3=3.2%


Rebuilt H1, PCA, year, and genome-tag evidence in 26.0s


## Part 2 — Joint model: how much of PC1 does ALL the evidence explain *together*?

Yesterday's checks were all marginal (one factor/tag at a time). leniency and contrarian_bias likely overlap with each other; the top genome tags may or may not add anything once the behavioral factors are accounted for. This fits one ridge regression on the combined feature set and reports cross-validated R² — the real answer to "how much of PC1 can we explain," not six separate partial answers.

Shuffled `KFold` used throughout (per yesterday's fix — `cohort_user_ids` is not randomly ordered).

In [3]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import cross_val_score, KFold

_t0 = time.time()
_cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Top-15 genome tags by |r| with PC1 (recomputed fresh, not hardcoded from yesterday)
pc1_c = pc1 - pc1.mean()
ugp_c = user_genome_profile - user_genome_profile.mean(axis=0)
num = pc1_c @ ugp_c
den = np.sqrt((pc1_c ** 2).sum()) * np.sqrt((ugp_c ** 2).sum(axis=0))
r_tags = np.asarray(num / np.where(den == 0, 1e-9, den)).flatten()
top15_idx = np.argsort(np.abs(r_tags))[::-1][:15]
top15_tag_names = [genome_tag_names[i] for i in top15_idx]
print("Top-15 genome tags going into the joint model:", top15_tag_names)

# Build the combined feature matrix
behavioral = np.column_stack([leniency, extremity, activity, genre_entropy, contrarian_bias, popularity_bias])
year_col = np.nan_to_num(user_avg_year, nan=np.nanmean(user_avg_year)).reshape(-1, 1)
tag_cols = user_genome_profile[:, top15_idx]

feature_blocks = {
    "behavioral only (6 factors)": behavioral,
    "behavioral + year": np.column_stack([behavioral, year_col]),
    "behavioral + year + top-15 genome tags (ALL evidence combined)": np.column_stack([behavioral, year_col, tag_cols]),
}

print("\n=== Joint ridge model: cross-validated R² explaining PC1 ===")
results = {}
for name, X in feature_blocks.items():
    Xz = (X - X.mean(axis=0)) / X.std(axis=0)
    model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])
    scores = cross_val_score(model, Xz, pc1, cv=_cv, scoring="r2")
    results[name] = scores.mean()
    print(f"  {name:55s} R² = {scores.mean():.3f}  (per-fold: {np.round(scores,3)})")

print(f"\nDone in {time.time()-_t0:.1f}s")

Top-15 genome tags going into the joint model: ['disappointing', 'graphic novel', 'too long', 'heroine in tight suit', 'waste of time', 'better than expected', 'product placement', 'crude humor', 'based on comic', 'tolkien', 'marvel', 'stupid as hell', 'watch the credits', 'southern theme', 'idiotic']

=== Joint ridge model: cross-validated R² explaining PC1 ===
  behavioral only (6 factors)                             R² = 0.294  (per-fold: [0.298 0.334 0.295 0.24  0.304])
  behavioral + year                                       R² = 0.321  (per-fold: [0.313 0.351 0.334 0.277 0.331])
  behavioral + year + top-15 genome tags (ALL evidence combined) R² = 0.416  (per-fold: [0.413 0.451 0.429 0.368 0.42 ])

Done in 0.1s


## Part 3 — Build a channel from PC1's combined signal

R²=0.416 (behavioral + year + genome tags combined) clears the bar — higher than PC2's single-factor r²=0.334. Fit the final ridge model on the full cohort, use its **predicted PC1 score per user** as the new channel's input signal (a composite of 6 behavioral factors + year + 15 genome tags, not any single guess), broadcast across each user's rated movies exactly like the personality/extremity channels, and train with the identical CD-1 procedure/hyperparameters.

In [4]:
_t0 = time.time()

# Fit the final combined model on the full cohort (not CV'd -- this is for generating
# the channel's input signal, not for reporting R²; R² was already honestly
# cross-validated in Part 2).
X_full = np.column_stack([behavioral, year_col, tag_cols])
X_full_z = (X_full - X_full.mean(axis=0)) / X_full.std(axis=0)
final_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]).fit(X_full_z, pc1)
pc1_predicted = final_model.predict(X_full_z)
print(f"Final model alpha={final_model.alpha_}, in-sample R²={final_model.score(X_full_z, pc1):.3f}")
print(f"pc1_predicted (raw scale): min={pc1_predicted.min():.3f} max={pc1_predicted.max():.3f} mean={pc1_predicted.mean():.3f}")

# Standardize before using as an RBM channel input: pc1's own natural scale is much
# larger than personality's (-2.24..1.59) or extremity's (0.31..2.03) -- an unscaled
# broadcast target that large is mostly outside what a sigmoid reconstruction can ever
# hit, which is why the first attempt's masked MSE (4.39) was ~100x the extremity
# channel's (0.045). z-score it to put it on the same order of magnitude.
pc1_signal = (pc1_predicted - pc1_predicted.mean()) / pc1_predicted.std()
print(f"pc1_signal (standardized): min={pc1_signal.min():.3f} max={pc1_signal.max():.3f} mean={pc1_signal.mean():.3f}")

# --- Build channelB2_pc1: broadcast pc1_signal across each user's rated movies ---
mask_arr = np.asarray(mask)
X2_pc1 = (mask_arr.astype(np.float32) * pc1_signal[:, None].astype(np.float32))
M2 = mask_arr.astype(np.float32)

# --- Same CD-1 training procedure as 17's Part 5c / 13b ---
N_HIDDEN, LR, BATCH_SIZE, EPOCHS, INIT_SEED = 128, 0.01, 500, 50, 42


def reconstruction_mse_b2(X, M_movies, W, b_h, chunk=250):
    n = X.shape[0]
    num, den = 0.0, 0.0
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n)
        x, m = X[i0:i1], M_movies[i0:i1]
        h_prob = sigmoid(x @ W + b_h)
        v_prob = sigmoid(h_prob @ W.T)
        num += float(((x - v_prob) ** 2 * m).sum())
        den += float(m.sum())
    return num / den if den > 0 else float("nan")


def train_rbm_b2(X, M_movies, n_hidden, seed, channel_name):
    rng = np.random.default_rng(seed)
    n_samples, n_visible = X.shape
    W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden)).astype(np.float64)
    b_h = np.zeros(n_hidden, dtype=np.float64)
    print(f"\n=== Training {channel_name} ===  samples={n_samples}, visible={n_visible}, hidden={n_hidden}")
    t0_all = time.time()
    for epoch in range(EPOCHS):
        t0 = time.time()
        epoch_delta = np.zeros_like(W)
        order = rng.permutation(n_samples)
        for start in range(0, n_samples, BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            v_data = X[idx].astype(np.float64)
            m_batch = M_movies[idx].astype(np.float64)
            bs = v_data.shape[0]
            h_prob = sigmoid(v_data @ W + b_h)
            h_data = (rng.random(h_prob.shape) < h_prob).astype(np.float64)
            v_recon_prob = sigmoid(h_data @ W.T) * m_batch
            v_recon = (rng.random(v_recon_prob.shape) < v_recon_prob).astype(np.float64) * m_batch
            h_recon_prob = sigmoid(v_recon @ W + b_h)
            h_recon = (rng.random(h_recon_prob.shape) < h_recon_prob).astype(np.float64)
            epoch_delta += LR * (v_data.T @ h_data - v_recon.T @ h_recon) / bs
            b_h += LR * (h_prob.mean(axis=0) - h_recon_prob.mean(axis=0))
        W += epoch_delta
        ep = epoch + 1
        if ep in (1, 25, EPOCHS):
            mse = reconstruction_mse_b2(X, M_movies, W, b_h)
            print(f"  Epoch {ep:02d} | masked recon MSE: {mse:.6f} | wall {time.time()-t0:.2f}s")
    print(f"  Done in {(time.time()-t0_all)/60:.1f} min")
    return W, b_h


W2_pc1, bh2_pc1 = train_rbm_b2(X2_pc1, M2, N_HIDDEN, INIT_SEED, "RBM_B2_pc1combined (discovered channel)")
np.save(proc / "rbmB2pc1_weights_5k.npy", W2_pc1)
np.save(proc / "rbmB2pc1_bias_hidden_5k.npy", bh2_pc1)
print(f"\nSaved rbmB2pc1_weights_5k.npy {W2_pc1.shape}. Total cell time: {time.time()-_t0:.1f}s")

Final model alpha=1.0, in-sample R²=0.424
pc1_predicted (raw scale): min=-11.770 max=7.953 mean=-0.000
pc1_signal (standardized): min=-5.197 max=3.512 mean=0.000



=== Training RBM_B2_pc1combined (discovered channel) ===  samples=5000, visible=13129, hidden=128


  Epoch 01 | masked recon MSE: 1.045906 | wall 2.88s


  Epoch 25 | masked recon MSE: 0.760187 | wall 2.40s


  Epoch 50 | masked recon MSE: 0.752790 | wall 2.89s
  Done in 1.4 min

Saved rbmB2pc1_weights_5k.npy (13129, 128). Total cell time: 86.5s


## Part 4 — Final test-set validation (5-way, first real contact with the 20% held-out set)

Everything above only touched the 80% train partition. This is the one place the 20% test set gets used, and only for final scoring. Five predictors, all evaluated on the same `test_labels.csv`, same metrics:

1. **Real RBM alone** (`r1`) — reused formula from `14b_evaluation_5k.ipynb`
2. **Existing hand-picked personality channel** (`r1 + v2`, leniency-based) — must reproduce 14b's numbers exactly (0.9423/0.7537 and 0.9027/0.6970) as a regression check
3. **Naive learned-bias baseline** (`r1` + a ridge-fit per-user bias from `mu_user`)
4. **Extremity channel** (`r1 + v2_ext`, from 17's Part 5c)
5. **PC1-combined channel** (`r1 + v2_pc1`, from this notebook's Part 3)

Overall + generous/strict/middle group breakdown, matching 14b's precedent.

In [5]:
_t0 = time.time()

# H2 for the existing personality channel, and rebuild the extremity channel's input
# (same construction as 17's Part 5c) so H2_ext can be computed here too.
X2_personality = np.asarray(channelB2, dtype=np.float32).reshape(n_users, -1)
H2 = sigmoid(X2_personality @ W2 + bh2)

X2_extremity = (mask_arr.astype(np.float32) * extremity[:, None].astype(np.float32))
H2_ext = sigmoid(X2_extremity @ W2ext + bh2ext)

H2_pc1 = sigmoid(X2_pc1 @ W2_pc1 + bh2_pc1)

# --- Map test rows -> (i, j) ---
test_df = test_labels.copy()
test_df["i"] = test_df["userId"].map(user_to_row)
test_df["j"] = test_df["movieId"].map(movie_to_col)
n_before = len(test_df)
test_df = test_df.dropna(subset=["i", "j"]).copy()
test_df["i"] = test_df["i"].astype(int)
test_df["j"] = test_df["j"].astype(int)
print(f"test rows usable: {len(test_df):,} / {n_before:,}")

i_all = test_df["i"].to_numpy()
j_all = test_df["j"].to_numpy()
y_true = test_df["rating"].to_numpy(dtype=np.float64)


def predict_r1_batch(i_arr, j_arr):
    """Softmax expected rating from B1 reconstruction (real part) -- unchanged, reused from 14b."""
    preds = np.empty(len(i_arr), dtype=np.float64)
    order = np.argsort(j_arr)
    j_sorted, i_sorted = j_arr[order], i_arr[order]
    start, n = 0, len(j_sorted)
    while start < n:
        j = int(j_sorted[start])
        end = start + 1
        while end < n and int(j_sorted[end]) == j:
            end += 1
        ii = i_sorted[start:end]
        W_slice = W1[j * K:(j + 1) * K]
        probs = sigmoid(H1[ii] @ W_slice.T)
        denom = np.where(probs.sum(axis=1) > 0, probs.sum(axis=1), 1.0)
        preds[order[start:end]] = (probs * RATING_LEVELS).sum(axis=1) / denom
        start = end
    return preds


def predict_v2_batch(H2_channel, W2_channel, i_arr, j_arr):
    """Second-channel reconstructed scalar at movie j for user i -- generalized to take any (H2, W2) pair."""
    preds = np.empty(len(i_arr), dtype=np.float64)
    order = np.argsort(j_arr)
    j_sorted, i_sorted = j_arr[order], i_arr[order]
    start, n = 0, len(j_sorted)
    while start < n:
        j = int(j_sorted[start])
        end = start + 1
        while end < n and int(j_sorted[end]) == j:
            end += 1
        ii = i_sorted[start:end]
        logits = H2_channel[ii] @ W2_channel[j]
        preds[order[start:end]] = sigmoid(logits)
        start = end
    return preds


print("Predicting r1 (Real RBM) ...")
r1 = predict_r1_batch(i_all, j_all)
print("Predicting v2 (existing personality channel) ...")
v2 = predict_v2_batch(H2, W2, i_all, j_all)
print("Predicting v2_ext (extremity channel) ...")
v2_ext = predict_v2_batch(H2_ext, W2ext, i_all, j_all)
print("Predicting v2_pc1 (PC1-combined channel) ...")
v2_pc1 = predict_v2_batch(H2_pc1, W2_pc1, i_all, j_all)

# Naive learned-bias baseline: ridge-fit a per-user bias from mu_user (train-only), add to r1
from sklearn.linear_model import Ridge
mu_user_train = user_means.reshape(-1, 1)
naive_model = Ridge(alpha=1.0).fit(mu_user_train, user_means - mu_global)
naive_bias_per_user = naive_model.predict(mu_user_train)
naive_bias = naive_bias_per_user[i_all]

predictions = {
    "Real RBM (B1 only)": np.clip(r1, 0.5, 5.0),
    "Existing personality (leniency)": np.clip(r1 + v2, 0.5, 5.0),
    "Naive learned bias": np.clip(r1 + naive_bias, 0.5, 5.0),
    "Extremity channel (5c)": np.clip(r1 + v2_ext, 0.5, 5.0),
    "PC1-combined channel (18)": np.clip(r1 + v2_pc1, 0.5, 5.0),
}

print(f"\nDone predicting in {time.time()-_t0:.1f}s")

test rows usable: 1,032,307 / 1,032,307
Predicting r1 (Real RBM) ...


Predicting v2 (existing personality channel) ...


Predicting v2_ext (extremity channel) ...


Predicting v2_pc1 (PC1-combined channel) ...



Done predicting in 4.6s


In [6]:
_t0 = time.time()

# --- Overall comparison table ---
rows = []
for name, pred in predictions.items():
    rmse = float(np.sqrt(np.mean((pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(pred - y_true)))
    rows.append({"model": name, "RMSE": rmse, "MAE": mae})
compare_df = pd.DataFrame(rows)
print("=== Overall comparison (5-way) ===")
print(compare_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# --- Regression check: rows 1-2 must reproduce 14b's existing numbers exactly ---
KNOWN_REAL_RMSE, KNOWN_REAL_MAE = 0.9423, 0.7537
KNOWN_HYP_RMSE, KNOWN_HYP_MAE = 0.9027, 0.6970
real_row = compare_df[compare_df.model == "Real RBM (B1 only)"].iloc[0]
hyp_row = compare_df[compare_df.model == "Existing personality (leniency)"].iloc[0]
real_ok = abs(real_row.RMSE - KNOWN_REAL_RMSE) < 0.001 and abs(real_row.MAE - KNOWN_REAL_MAE) < 0.001
hyp_ok = abs(hyp_row.RMSE - KNOWN_HYP_RMSE) < 0.001 and abs(hyp_row.MAE - KNOWN_HYP_MAE) < 0.001
print(f"\nRegression check vs 14b: Real RBM {'PASS' if real_ok else 'FAIL'} "
      f"(got {real_row.RMSE:.4f}/{real_row.MAE:.4f}, expected {KNOWN_REAL_RMSE}/{KNOWN_REAL_MAE})")
print(f"Regression check vs 14b: Existing personality {'PASS' if hyp_ok else 'FAIL'} "
      f"(got {hyp_row.RMSE:.4f}/{hyp_row.MAE:.4f}, expected {KNOWN_HYP_RMSE}/{KNOWN_HYP_MAE})")
assert real_ok and hyp_ok, "Regression check failed -- something in data loading changed vs 14b. Do not trust the new comparators until this passes."

# --- Group breakdown (generous / strict / middle), matching 14b's precedent ---
def group_label(mu):
    if mu > 4.0:
        return "generous"
    if mu < 2.5:
        return "strict"
    return "middle"

user_group = np.array([group_label(mu) for mu in user_means])
test_df["group"] = user_group[i_all]
test_df["y_true"] = y_true
for name, pred in predictions.items():
    test_df[name] = pred

group_rows = []
for g in ("generous", "strict", "middle", "all"):
    sub = test_df if g == "all" else test_df[test_df["group"] == g]
    row = {"group": g, "n_users": int(np.sum(user_group == g)) if g != "all" else n_users, "n_test_ratings": len(sub)}
    for name in predictions:
        yt, pr = sub["y_true"].to_numpy(), sub[name].to_numpy()
        row[f"{name}_RMSE"] = float(np.sqrt(np.mean((pr - yt) ** 2)))
    group_rows.append(row)
group_df = pd.DataFrame(group_rows)
print("\n=== Group breakdown (RMSE) ===")
print(group_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

compare_df.to_csv(out_dir / "ablation_hyperbolic_vs_bias_5k.csv", index=False)
group_df.to_csv(out_dir / "ablation_group_breakdown_5k.csv", index=False)
print(f"\nSaved ablation_hyperbolic_vs_bias_5k.csv, ablation_group_breakdown_5k.csv")
print(f"Done in {time.time()-_t0:.1f}s")

=== Overall comparison (5-way) ===
                          model   RMSE    MAE
             Real RBM (B1 only) 0.9423 0.7537
Existing personality (leniency) 0.9027 0.6970
             Naive learned bias 0.9138 0.7148
         Extremity channel (5c) 1.1744 0.9110
      PC1-combined channel (18) 0.9240 0.7183

Regression check vs 14b: Real RBM PASS (got 0.9423/0.7537, expected 0.9423/0.7537)
Regression check vs 14b: Existing personality PASS (got 0.9027/0.6970, expected 0.9027/0.697)



=== Group breakdown (RMSE) ===
   group  n_users  n_test_ratings  Real RBM (B1 only)_RMSE  Existing personality (leniency)_RMSE  Naive learned bias_RMSE  Extremity channel (5c)_RMSE  PC1-combined channel (18)_RMSE
generous      236           43090                   1.1907                                0.9085                   0.8226                       0.8610                          0.9085
  strict      145           34176                   1.2731                                1.3010                   1.3233                       1.8014                          1.2870
  middle     4619          955041                   0.9153                                0.8849                   0.8997                       1.1582                          0.9090
     all     5000         1032307                   0.9423                                0.9027                   0.9138                       1.1744                          0.9240

Saved ablation_hyperbolic_vs_bias_5k.csv, ablation_g